# EnerGIS - Quick Test

Minimales Notebook zum Testen der Rolling Horizon Optimierung.

**Aktueller Test:**
- `enforce: true` → Terminal Constraints AKTIVIERT
- `fix_design: true` → Investment nach 1. Window fixiert
- `thermal_network: false` → Temporär deaktiviert

In [ ]:
# Setup
import sys
from pathlib import Path

current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break

from energis.run import rolling_horizon as rh
print(f"✅ Project: {PROJECT_ROOT}")

In [ ]:
# Konfiguration
CONFIG = ['configs/stadtbach.yaml']

print(f"📋 Config: {CONFIG[0]}")
print(f"   Exists: {(PROJECT_ROOT / CONFIG[0]).exists()}")

In [ ]:
%%time
# Optimierung ausführen
print("🚀 STARTE OPTIMIERUNG\n" + "="*70)

try:
    workflow = rh.run_workflow(CONFIG)
    print("\n" + "="*70)
    print("✅ ERFOLGREICH")
    SUCCESS = True
except Exception as e:
    print(f"\n❌ FEHLER: {e}")
    import traceback
    traceback.print_exc()
    SUCCESS = False

In [ ]:
# Ergebnisse
if SUCCESS:
    from energis.io.notebook_helpers import display_workflow_summary, display_kpi_summary
    
    display_workflow_summary(workflow)
    print("\n")
    display_kpi_summary(workflow)
else:
    print("⚠️ Keine Ergebnisse verfügbar")

In [ ]:
# Speicher-Check
if SUCCESS and workflow:
    result = workflow.rh_result or workflow.pf_result
    
    if 'TES_SOC_MWh' in result.series:
        import matplotlib.pyplot as plt
        import pandas as pd
        
        soc = result.series['TES_SOC_MWh']
        
        print(f"📊 Speicher SOC Analyse:")
        print(f"   Min:  {min(soc):.1f} MWh")
        print(f"   Max:  {max(soc):.1f} MWh")
        print(f"   Mean: {sum(soc)/len(soc):.1f} MWh")
        print(f"   Std:  {pd.Series(soc).std():.1f} MWh")
        
        plt.figure(figsize=(12, 4))
        plt.plot(soc, linewidth=1)
        plt.ylabel('SOC [MWh]')
        plt.title('Speicher State of Charge')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Check if cycling
        if pd.Series(soc).std() > 10:
            print("\n✅ Speicher cycelt (std > 10 MWh)")
        else:
            print("\n⚠️ Speicher inaktiv (std < 10 MWh)")
    else:
        print("⚠️ Keine SOC Daten verfügbar")